<a href="https://colab.research.google.com/github/Rais-Ataullov/BigData/blob/lab2/L2_Reports_with_Apache_Spark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
import pyspark.sql as sql

conf = SparkConf().setAppName("L2_Reports_with_Apache_Spark").setMaster('local[*]')

sc = SparkContext(conf=conf)
spark = SparkSession(sc)

POSTS_XML = "posts_sample.xml"
LANGS_CSV = "programming-languages.csv"
PARQUET_OUT = "top_languages_2010_2020.parquet"

In [2]:
langsData = spark.read\
.option("header", True)\
.option("inferSchema", True)\
.csv(LANGS_CSV).distinct()

langsData.createOrReplaceTempView('langs')

langsData = spark.sql("""
    SELECT TRIM(LOWER(CAST(langs.name AS STRING))) AS language FROM langs
""")

langsData.createOrReplaceTempView('langs')
langsData.show()

+-----------------+
|         language|
+-----------------+
|            tutor|
|    orca/modula-2|
|            alice|
|             amos|
|              tal|
|         ubercode|
|               a+|
|              mpd|
|          netlogo|
|            t-sql|
|            arexx|
|            dylan|
|            gnu e|
|pascal – iso 7185|
|             pike|
|              go!|
|              rpl|
|          l# .net|
|            oriel|
|              sed|
+-----------------+
only showing top 20 rows


In [3]:
stackOverflow_reportsData = spark.read.format("xml").option("rowTag", "row") \
    .option("attributePrefix", "").load(POSTS_XML)

stackOverflow_reportsData.show()
stackOverflow_reportsData.createOrReplaceTempView("xml")

+----------------+-----------+--------------------+----------+------------+--------------------+--------------------+-------------+-------+--------------------+--------------------+---------------------+----------------+----------------+-----------+--------+----------+-----+--------------------+--------------------+---------+
|AcceptedAnswerId|AnswerCount|                Body|ClosedDate|CommentCount|  CommunityOwnedDate|        CreationDate|FavoriteCount|     Id|    LastActivityDate|        LastEditDate|LastEditorDisplayName|LastEditorUserId|OwnerDisplayName|OwnerUserId|ParentId|PostTypeId|Score|                Tags|               Title|ViewCount|
+----------------+-----------+--------------------+----------+------------+--------------------+--------------------+-------------+-------+--------------------+--------------------+---------------------+----------------+----------------+-----------+--------+----------+-----+--------------------+--------------------+---------+
|               

In [4]:
parsed_posts = spark.sql("""
    SELECT
        CAST(SUBSTRING(xml.CreationDate, 1, 4) AS INT) AS year,
        REGEXP_EXTRACT(xml.Tags, '<([^>]+)>', 1) AS tag
    FROM xml
""").dropna().filter("year >= 2010 AND year <= 2020")

print(f"Найдено постов: {parsed_posts.count()}")
parsed_posts.show(truncate=False)

parsed_posts.createOrReplaceTempView("parsed_posts")

Найдено постов: 17642
+----+------------------+
|year|tag               |
+----+------------------+
|2010|c++               |
|2010|sharepoint        |
|2010|iphone            |
|2010|symfony1          |
|2010|java              |
|2010|visual-studio-2010|
|2010|cakephp           |
|2010|git               |
|2010|drupal            |
|2010|php               |
|2010|c#                |
|2010|c#                |
|2010|sql               |
|2010|.htaccess         |
|2010|wcf               |
|2010|mod-rewrite       |
|2010|sql               |
|2010|ruby              |
|2010|android           |
|2010|iphone            |
+----+------------------+
only showing top 20 rows


In [5]:
lang_usage = spark.sql("""
    SELECT
        t.year,
        l.language
    FROM parsed_posts t
    INNER JOIN langs l ON t.tag = l.language
""")

# lang_usage.show()

lang_usage.createOrReplaceTempView("lang_usage")

stats = spark.sql("""
    SELECT
        year,
        language,
        COUNT(*) AS count
    FROM lang_usage
    GROUP BY year, language
""")

stats.show()

+----+-----------+-----+
|year|   language|count|
+----+-----------+-----+
|2017|       perl|    6|
|2019| typescript|    2|
|2011|    haskell|    1|
|2012|       bash|    5|
|2018|       glsl|    1|
|2011|objective-c|   15|
|2013|        php|  190|
|2013|       chef|    1|
|2013|     delphi|    7|
|2016| coldfusion|    4|
|2016|        awk|    3|
|2019|         go|    8|
|2015|       bash|    9|
|2013|          c|   28|
|2016|         f#|    2|
|2013|    haskell|    2|
|2014|         f#|    1|
|2013|     matlab|    7|
|2015|        coq|    1|
|2016|     matlab|   14|
+----+-----------+-----+
only showing top 20 rows


In [6]:
stats.createOrReplaceTempView("stats")

top_10_report = spark.sql("""
    SELECT
    year,
    language,
    count,
    rank
FROM (
    SELECT
        year,
        language,
        count,
        ROW_NUMBER() OVER (PARTITION BY year ORDER BY count DESC, language ASC) AS rank
    FROM stats
) t
WHERE rank <= 10
ORDER BY year, rank
""")

print("Финальный отчет (фрагмент):")
top_10_report.show(25, truncate=False)

Финальный отчет (фрагмент):
+----+-----------+-----+----+
|year|language   |count|rank|
+----+-----------+-----+----+
|2010|java       |51   |1   |
|2010|php        |46   |2   |
|2010|javascript |37   |3   |
|2010|python     |24   |4   |
|2010|c          |16   |5   |
|2010|delphi     |7    |6   |
|2010|objective-c|7    |7   |
|2010|ruby       |5    |8   |
|2010|bash       |2    |9   |
|2010|perl       |2    |10  |
|2011|php        |100  |1   |
|2011|java       |91   |2   |
|2011|javascript |77   |3   |
|2011|python     |37   |4   |
|2011|c          |21   |5   |
|2011|objective-c|15   |6   |
|2011|ruby       |11   |7   |
|2011|bash       |5    |8   |
|2011|delphi     |5    |9   |
|2011|perl       |4    |10  |
|2012|php        |151  |1   |
|2012|java       |121  |2   |
|2012|javascript |115  |3   |
|2012|python     |67   |4   |
|2012|objective-c|29   |5   |
+----+-----------+-----+----+
only showing top 25 rows


In [7]:
top_10_report.write.mode("overwrite").parquet(PARQUET_OUT)
print(f"Данные сохранены в {PARQUET_OUT}")

Данные сохранены в top_languages_2010_2020.parquet


In [8]:
top_10_report.createOrReplaceTempView('top_10_report')

pivot_report = spark.sql("""
    SELECT
        year,
        MAX(CASE WHEN rank = 1 THEN language END) AS rank_1,
        MAX(CASE WHEN rank = 2 THEN language END) AS rank_2,
        MAX(CASE WHEN rank = 3 THEN language END) AS rank_3,
        MAX(CASE WHEN rank = 4 THEN language END) AS rank_4,
        MAX(CASE WHEN rank = 5 THEN language END) AS rank_5,
        MAX(CASE WHEN rank = 6 THEN language END) AS rank_6,
        MAX(CASE WHEN rank = 7 THEN language END) AS rank_7,
        MAX(CASE WHEN rank = 8 THEN language END) AS rank_8,
        MAX(CASE WHEN rank = 9 THEN language END) AS rank_9,
        MAX(CASE WHEN rank = 10 THEN language END) AS rank_10
    FROM top_10_report
    GROUP BY year
    ORDER BY year
""")

print("Pivot таблица (топ языков по годам):")
pivot_report.show(truncate=False)

Pivot таблица (топ языков по годам):
+----+----------+----------+----------+------+-----------+-----------+-----------+------+----------+-----------+
|year|rank_1    |rank_2    |rank_3    |rank_4|rank_5     |rank_6     |rank_7     |rank_8|rank_9    |rank_10    |
+----+----------+----------+----------+------+-----------+-----------+-----------+------+----------+-----------+
|2010|java      |php       |javascript|python|c          |delphi     |objective-c|ruby  |bash      |perl       |
|2011|php       |java      |javascript|python|c          |objective-c|ruby       |bash  |delphi    |perl       |
|2012|php       |java      |javascript|python|objective-c|c          |ruby       |r     |bash      |scala      |
|2013|java      |php       |javascript|python|c          |r          |objective-c|ruby  |scala     |perl       |
|2014|javascript|java      |php       |python|c          |r          |objective-c|matlab|bash      |ruby       |
|2015|javascript|java      |php       |python|r          |c

In [9]:
import shutil
from google.colab import files

shutil.make_archive("report_parquet", 'zip', PARQUET_OUT)
files.download("report_parquet.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
sc.stop()